# Homework 4: Model Building and Tracking

Using Random Forest to predict F1 race positions, tracked with MLflow.

In [0]:
import mlflow
import sklearn
print(f"MLflow version: {mlflow.__version__}")
print(f"Sklearn version: {sklearn.__version__}")
print("Environment ready!")

## Data Loading

In [0]:
import pandas as pd
import numpy as np

path = "/Volumes/gr5069/raw/f1_data/"

results    = spark.read.csv(path + "results.csv",          header=True, inferSchema=True).toPandas()
races      = spark.read.csv(path + "races.csv",            header=True, inferSchema=True).toPandas()
drivers    = spark.read.csv(path + "driver_standings.csv", header=True, inferSchema=True).toPandas()
qualifying = spark.read.csv(path + "qualifying.csv",       header=True, inferSchema=True).toPandas()

print(f"results:    {results.shape}")
print(f"races:      {races.shape}")
print(f"drivers:    {drivers.shape}")
print(f"qualifying: {qualifying.shape}")
print("Data loaded!")

## Data Preparation

In [0]:
df = results.merge(races[["raceId", "year", "round", "circuitId"]], on="raceId", how="left")

qualifying_renamed = qualifying[["raceId", "driverId", "q1", "q2", "q3", "position"]].copy()
qualifying_renamed = qualifying_renamed.rename(columns={"position": "quali_position"})

df = df.merge(qualifying_renamed, on=["raceId", "driverId"], how="left")

df = df[["grid", "laps", "milliseconds", "fastestLap", "fastestLapSpeed",
         "year", "round", "quali_position", "positionOrder"]].copy()

df = df.replace(r'\\N', np.nan, regex=True)
df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna()

X = df[["grid", "laps", "milliseconds", "fastestLap", "fastestLapSpeed", "year", "round", "quali_position"]]
y = df["positionOrder"]

print(f"Dataset shape: {X.shape}")
print(f"Target range: {y.min()} to {y.max()}")
print("Ready!")

## Train/Test Split

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## MLflow Experiment

I run 10 experiments with different hyperparameter combinations for Random Forest. For each run I log:
- **Parameters**: n_estimators, max_depth, min_samples_split, min_samples_leaf
- **Metrics**: RMSE, MAE, R2
- **Model**: the trained Random Forest
- **Artifacts**: feature importance plot and actual vs predicted plot

In [0]:
mlflow.end_run()

experiment_name = "/Users/yk3167@columbia.edu/F1_Position_Prediction"
mlflow.set_experiment(experiment_name)

param_grid = [
    {"n_estimators": 50,  "max_depth": 5,    "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10,   "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 10,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15,   "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 20,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 20,   "min_samples_split": 10,"min_samples_leaf": 4},
    {"n_estimators": 500, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
]

results_list = []

for i, params in enumerate(param_grid):
    with mlflow.start_run(run_name=f"RF_run_{i+1}"):
        
        rf = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        for k, v in params.items():
            mlflow.log_param(k, v)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        mlflow.sklearn.log_model(rf, "model")

        results_list.append({**params, "rmse": rmse, "mae": mae, "r2": r2})
        print(f"Run {i+1} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

print("\nAll runs complete!")

## Results Summary

In [0]:
results_df = pd.DataFrame(results_list).sort_values("rmse")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\nBest run: n_estimators={int(best.n_estimators)}, max_depth={best.max_depth}, RMSE={best.rmse:.4f}, R2={best.r2:.4f}")

## Artifacts - Best Model

In [0]:
best_params = {
    "n_estimators": 300,
    "max_depth": 20,
    "min_samples_split": 10,
    "min_samples_leaf": 4
}

rf_best = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
y_pred_best = rf_best.predict(X_test)

feature_names = X.columns.tolist()

with mlflow.start_run(run_name="RF_best_model"):
    # Log params and metrics
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, y_pred_best)))
    mlflow.log_metric("mae",  mean_absolute_error(y_test, y_pred_best))
    mlflow.log_metric("r2",   r2_score(y_test, y_pred_best))
    mlflow.sklearn.log_model(rf_best, "model")

    # Artifact 1: Feature Importance
    fig, ax = plt.subplots(figsize=(8, 5))
    importance = pd.Series(rf_best.feature_importances_, index=feature_names).sort_values()
    importance.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Feature Importance - Best Model")
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig("/tmp/feature_importance.png")
    mlflow.log_artifact("/tmp/feature_importance.png")
    plt.show()

    # Artifact 2: Actual vs Predicted
    fig2, ax2 = plt.subplots(figsize=(6, 6))
    ax2.scatter(y_test, y_pred_best, alpha=0.4, color="coral")
    ax2.plot([1, 20], [1, 20], "r--", label="Perfect Prediction")
    ax2.set_xlabel("Actual Position")
    ax2.set_ylabel("Predicted Position")
    ax2.set_title("Actual vs Predicted - Best Model")
    ax2.legend()
    plt.tight_layout()
    plt.savefig("/tmp/actual_vs_predicted.png")
    mlflow.log_artifact("/tmp/actual_vs_predicted.png")
    plt.show()

print("Artifacts logged!")

## Best Model Selection

**Best run: n_estimators=300, max_depth=20, min_samples_split=10, min_samples_leaf=4**

| Metric | Value |
|--------|-------|
| RMSE   | 2.7274 |
| MAE    | 2.0279 |
| R2     | 0.5235 |

This model performed best across all 10 runs because:
1. Lowest RMSE (2.7274) — predicted finishing position is off by ~2.7 places on average
2. Highest R2 (0.5235) — explains 52% of variance in race finishing positions
3. Deeper trees (max_depth=20) with min_samples_leaf=4 balance complexity and generalization
4. Shallower runs (max_depth=5) underfit slightly, while unconstrained depth (None) did not improve performance